# LearnMate — Qwen3-4B RAG API (Kaggle GPU)


In [1]:
# Cell 1 — Install all dependencies.
%pip install -q "transformers>=4.51.0,<5" "accelerate>=1.0.0" safetensors fastapi uvicorn pyngrok langchain-core langchain-text-splitters langchain-chroma langchain-huggingface sentence-transformers pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 79.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 64.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 86.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.1 MB/s eta 0:00:0

In [2]:
# Cell 2 — Check the GPU.
import torch
import platform

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

    gpu_memory = torch.cuda.get_device_properties(0).total_memory
    print(f"GPU memory: {gpu_memory / 1024**3:.2f} GB")

else:
    print("No GPU detected. Enable a GPU in Settings > Accelerator, then restart the session.")

Python version: 3.13.15
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU name: Tesla T4
GPU memory: 14.56 GB


In [3]:
# Cell 3 — Load Qwen.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen3-4B-Instruct-2507"

if globals().get("model") is not None and globals().get("tokenizer") is not None:
    print("Model already loaded on", model.device, "- skipping reload.")
else:
    if not torch.cuda.is_available():
        print(
            "WARNING: no GPU detected. Enable a GPU in Settings > Accelerator "
            "and restart the session; the model is far too slow on CPU."
        )

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    ).eval()

    print("Model loaded on", model.device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded on cuda:0


In [4]:
# Cell 4 — Text generation .
import threading
import torch

# The API runs requests in worker threads; only one may use the GPU model at a time.
GENERATION_LOCK = globals().get("GENERATION_LOCK") or threading.Lock()


def generate_text(
    prompt,
    max_length=300,
    system_prompt=None,
    do_sample=None,
    repetition_penalty=None,
):
    """Generate a reply from Qwen.

    generate_text(prompt, max_length) behaves as before (model's default
    sampling). The API passes the extra options for focused, less
    repetitive answers.
    """
    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": prompt})

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        enable_thinking=False,
        return_dict=True
    ).to(model.device)

    generation_kwargs = {
        "max_new_tokens": max_length,
        "pad_token_id": tokenizer.eos_token_id,
    }

    if do_sample is not None:
        generation_kwargs["do_sample"] = do_sample

        if not do_sample:
            # Clear the sampling defaults from Qwen's generation config to avoid warnings.
            generation_kwargs.update(temperature=None, top_p=None, top_k=None)

    if repetition_penalty is not None:
        generation_kwargs["repetition_penalty"] = repetition_penalty

    with GENERATION_LOCK, torch.inference_mode():
        outputs = model.generate(**inputs, **generation_kwargs)

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("generate_text is ready.")

generate_text is ready.


In [9]:
# Cell 5 — Secrets.
import secrets
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()


def read_kaggle_secret(name):
    try:
        return user_secrets.get_secret(name) or None
    except Exception:
        return None


NGROK_AUTHTOKEN = read_kaggle_secret("NGROK_AUTHTOKEN")

if not NGROK_AUTHTOKEN:
    raise RuntimeError(
        "Kaggle secret NGROK_AUTHTOKEN is missing. Add it in Add-ons > Secrets "
        "and enable it for this notebook."
    )

print("ngrok token ready:", True)

_secret_api_key = read_kaggle_secret("LEARNMATE_API_KEY")

if _secret_api_key:
    LEARNMATE_API_KEY = _secret_api_key
    print("Using LEARNMATE_API_KEY from Kaggle secrets (same key every session).")
elif globals().get("LEARNMATE_API_KEY"):
    print("Keeping the temporary API key created earlier in this session.")
else:
    LEARNMATE_API_KEY = secrets.token_urlsafe(32)
    print("No LEARNMATE_API_KEY secret found, so a temporary key was created for this session:")
    print(LEARNMATE_API_KEY)
    print(
        "Copy it into learnmate/.env.local, then CLEAR THIS CELL'S OUTPUT before saving or "
        "sharing the notebook. Adding a LEARNMATE_API_KEY Kaggle secret avoids this step."
    )

ngrok token ready: True
Using LEARNMATE_API_KEY from Kaggle secrets (same key every session).


In [10]:
# Cell 6 — RAG pipeline: PDF -> cleaned pages -> chunks (with page numbers) -> Chroma.
import io
import random
import re
import uuid
from collections import Counter
from pathlib import Path

import numpy as np
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Store the vector database in Kaggle's working directory.
CHROMA_DIR = "/kaggle/working/learnmate_chroma"

# Lightweight embedding model for English lecture notes.
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

MAX_PDF_BYTES = 20 * 1024 * 1024
MIN_PAGE_CHARS = 20          # pages with less text are treated as image-only
MIN_CHUNK_CHARS = 40         # drop tiny fragments such as lone titles
RETRIEVAL_CANDIDATES = 12    # chunks fetched from Chroma before re-ranking
RETRIEVAL_TOP_K = 5          # chunks passed to the model
KEYWORD_WEIGHT = 0.15        # bonus for chunks containing the question's key terms
MIN_RELEVANCE = 0.30         # below this the evidence is treated as weak
NO_MATCH_RELEVANCE = 0.12    # below this the lecture is treated as not covering the question
DOCUMENT_ID_PATTERN = re.compile(r"[A-Za-z0-9-]{8,64}")

STOPWORDS = {
    "what", "which", "when", "where", "does", "how", "why", "who", "whom", "the", "this",
    "that", "these", "those", "with", "from", "into", "about", "between", "explain",
    "describe", "define", "difference", "lecture", "please", "tell", "give", "there",
    "their", "they", "them", "have", "has", "been", "being", "were", "will", "would",
    "should", "could", "your", "mean", "means", "used", "using", "role", "work", "works",
}

if globals().get("embeddings") is None:
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
)

# Keep lectures uploaded earlier in this kernel when the cell is re-run.
lecture_stores = globals().get("lecture_stores") or {}
lecture_info = globals().get("lecture_info") or {}


class PdfProcessingError(Exception):
    """A user-facing PDF problem with the HTTP status code to return."""

    def __init__(self, message, status_code=400):
        super().__init__(message)
        self.status_code = status_code


def _normalize(text):
    return re.sub(r"\s+", " ", text).strip().lower()


def remove_repeated_lines(page_texts):
    """Remove header/footer lines (address, author, logo text) repeated on most pages."""
    if len(page_texts) < 5:
        return page_texts

    line_counts = Counter()

    for text in page_texts:
        line_counts.update({_normalize(line) for line in text.splitlines() if _normalize(line)})

    threshold = max(3, int(len(page_texts) * 0.6))
    repeated = {line for line, count in line_counts.items() if count >= threshold}

    return [
        "\n".join(line for line in text.splitlines() if _normalize(line) not in repeated)
        for text in page_texts
    ]


def clean_page_text(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()


def extract_pdf_pages(pdf_bytes):
    """Return (total_pages, [(page_number, text), ...]) or raise PdfProcessingError."""
    if not pdf_bytes:
        raise PdfProcessingError("The uploaded file is empty.")

    if len(pdf_bytes) > MAX_PDF_BYTES:
        raise PdfProcessingError("The PDF is larger than 20 MB.", status_code=413)

    if b"%PDF" not in pdf_bytes[:1024]:
        raise PdfProcessingError("The uploaded file is not a valid PDF.")

    try:
        reader = PdfReader(io.BytesIO(pdf_bytes))

        if reader.is_encrypted:
            try:
                decrypted = reader.decrypt("")
            except Exception:
                decrypted = 0

            if not decrypted:
                raise PdfProcessingError(
                    "This PDF is password-protected. Remove the password and upload it again."
                )

        total_pages = len(reader.pages)
    except PdfProcessingError:
        raise
    except Exception as error:
        raise PdfProcessingError("The PDF could not be read. It may be corrupted.") from error

    if total_pages == 0:
        raise PdfProcessingError("The PDF has no pages.")

    raw_texts = []

    for page in reader.pages:
        try:
            raw_texts.append(page.extract_text() or "")
        except Exception:
            raw_texts.append("")  # one broken page should not fail the whole upload

    cleaned = [clean_page_text(text) for text in remove_repeated_lines(raw_texts)]

    pages = [
        (page_number, text)
        for page_number, text in enumerate(cleaned, start=1)
        if len(text) >= MIN_PAGE_CHARS
    ]

    if not pages:
        raise PdfProcessingError(
            "No selectable text was found. This PDF seems to contain scanned images, "
            "which are not supported.",
            status_code=422,
        )

    return total_pages, pages


def _open_store(document_id):
    return Chroma(
        collection_name=f"lecture_{document_id}",
        embedding_function=embeddings,
        persist_directory=CHROMA_DIR,
    )


def index_pdf_bytes(pdf_bytes, filename="lecture.pdf", document_id=None):
    """Extract, chunk and index a PDF. Returns upload statistics."""
    total_pages, pages = extract_pdf_pages(pdf_bytes)

    documents = [
        Document(page_content=text, metadata={"page": page_number, "source": filename})
        for page_number, text in pages
    ]

    chunks = [
        chunk for chunk in splitter.split_documents(documents)
        if len(chunk.page_content.strip()) >= MIN_CHUNK_CHARS
    ]

    if not chunks:
        raise PdfProcessingError(
            "The PDF contains too little text to study from.", status_code=422
        )

    document_id = document_id or uuid.uuid4().hex

    for index, chunk in enumerate(chunks):
        chunk.metadata["chunk_index"] = index

    store = _open_store(document_id)
    store.add_documents(chunks, ids=[f"{document_id}-{i}" for i in range(len(chunks))])

    lecture_stores[document_id] = store
    lecture_info[document_id] = {"filename": filename, "pages": total_pages}

    return {
        "document_id": document_id,
        "filename": filename,
        "pages": total_pages,
        "readable_pages": len(pages),
        "chunks": len(chunks),
    }


def get_lecture_store(document_id):
    """Find a lecture in memory, or reload it from disk after a kernel restart."""
    store = lecture_stores.get(document_id)

    if store is not None:
        return store

    if not DOCUMENT_ID_PATTERN.fullmatch(document_id or ""):
        return None

    store = _open_store(document_id)

    if store._collection.count() == 0:
        store.delete_collection()  # opening it created an empty collection
        return None

    lecture_stores[document_id] = store
    return store


def _keywords(question):
    words = re.findall(r"[a-zA-Z][a-zA-Z0-9-]{3,}", question.lower())
    return {word for word in words if word not in STOPWORDS}


def _jaccard(a, b):
    return len(a & b) / len(a | b) if a and b else 0.0


def retrieve_chunks(document_id, question, top_k=RETRIEVAL_TOP_K):
    """Return (chunks, confidence) where confidence is 'high', 'low' or 'none'.

    Chunks are re-ranked by cosine similarity plus a small keyword bonus,
    near-duplicates are removed, and chunks far below the best match are dropped.
    """
    store = get_lecture_store(document_id)

    if store is None:
        raise LookupError(document_id)

    candidates = store.similarity_search(question, k=RETRIEVAL_CANDIDATES)

    if not candidates:
        return [], "none"

    question_vector = np.array(embeddings.embed_query(question))
    chunk_vectors = np.array(embeddings.embed_documents([c.page_content for c in candidates]))
    similarities = chunk_vectors @ question_vector

    terms = _keywords(question)
    scored = []

    for doc, similarity in zip(candidates, similarities):
        bonus = 0.0

        if terms:
            text = doc.page_content.lower()
            bonus = KEYWORD_WEIGHT * sum(term in text for term in terms) / len(terms)

        scored.append((float(similarity) + bonus, doc))

    scored.sort(key=lambda item: item[0], reverse=True)
    best_score = scored[0][0]

    if best_score < NO_MATCH_RELEVANCE:
        return [], "none"

    confidence = "high" if best_score >= MIN_RELEVANCE else "low"
    cutoff = max(MIN_RELEVANCE, best_score - 0.25) if confidence == "high" else NO_MATCH_RELEVANCE
    limit = top_k if confidence == "high" else 2

    selected = []
    seen_word_sets = []

    for score, doc in scored:
        if score < cutoff or len(selected) >= limit:
            break

        words = set(_normalize(doc.page_content).split())

        if any(_jaccard(words, seen) > 0.8 for seen in seen_word_sets):
            continue  # near-duplicate (e.g. chunk overlap or repeated slide)

        seen_word_sets.append(words)
        selected.append({
            "text": doc.page_content.strip(),
            "page": doc.metadata.get("page"),
            "chunk_index": doc.metadata.get("chunk_index", 0),
            "score": round(score, 3),
        })

    return selected, confidence


def sample_lecture_chunks(document_id, max_chunks=8, max_chars=6000):
    """Pick chunks spread across the whole lecture (used for quizzes without a topic)."""
    store = get_lecture_store(document_id)

    if store is None:
        raise LookupError(document_id)

    data = store.get(include=["documents", "metadatas"])
    items = [
        {"text": text.strip(), "page": (meta or {}).get("page"),
         "chunk_index": (meta or {}).get("chunk_index", 0)}
        for text, meta in zip(data["documents"], data["metadatas"])
        if text and text.strip()
    ]

    substantial = [item for item in items if len(item["text"]) >= 150]
    items = substantial or items

    if len(items) > max_chunks:
        items = random.sample(items, max_chunks)

    items.sort(key=lambda item: item["chunk_index"])

    selected, total = [], 0

    for item in items:
        if total + len(item["text"]) > max_chars and selected:
            break
        selected.append(item)
        total += len(item["text"])

    return selected


def format_context(chunks):
    ordered = sorted(chunks, key=lambda item: item["chunk_index"])
    return "\n\n".join(f"[Page {item['page']}]\n{item['text']}" for item in ordered)


# Notebook helpers (same names as before).
def process_pdf(pdf_path, document_id=None):
    """Index a PDF from disk, e.g. one attached under /kaggle/input."""
    pdf_path = Path(pdf_path)

    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    return index_pdf_bytes(pdf_path.read_bytes(), pdf_path.name, document_id)


def search_lecture(document_id, question, k=4):
    """Retrieve the most relevant chunks from one lecture."""
    chunks, _ = retrieve_chunks(document_id, question, top_k=k)
    return [{"text": item["text"], "page": item["page"]} for item in chunks]


print("LearnMate RAG is ready. Lectures in memory:", len(lecture_stores))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LearnMate RAG is ready. Lectures in memory: 0


In [11]:
# Cell 7 — FastAPI app.
import json
import random
import re
import secrets
import traceback
from typing import Optional

import torch
from fastapi import Depends, FastAPI, File, Header, HTTPException, Request, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field, field_validator

app = FastAPI(title="LearnMate Qwen API", version="1.1")

# Only needed if the browser calls the API directly; the Vite proxy does not need CORS.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173", "http://127.0.0.1:5173"],
    allow_methods=["GET", "POST"],
    allow_headers=["Content-Type", "X-API-Key", "ngrok-skip-browser-warning"],
)

LECTURE_NOT_FOUND = (
    "Lecture not found. The Kaggle session may have restarted - please upload the PDF again."
)

TUTOR_SYSTEM_PROMPT = """You are LearnMate, a careful and friendly AI tutor. You help a student understand THEIR uploaded lecture.

Rules:
1. Answer only the student's exact question. Do not drift into other topics.
2. Use ONLY facts stated in the lecture excerpts. Do not add outside knowledge, even if you know the topic.
3. Keep related concepts separate: never transfer a property, mechanism, value or step from one concept, device or procedure to another. When the excerpts describe similar concepts, make clear which one each statement is about.
4. If the excerpts do not contain enough information, say so plainly (for example: "The lecture does not explain ..."), then give only what the excerpts do support. Never guess or invent details, numbers or citations.
5. Cite the page of each key point as [Page X], using only page numbers that appear in the excerpts.
6. Be concise and student-friendly: start with a direct 1-2 sentence answer, then at most 4 short bullet points if they help. Usually stay under 150 words. Do not repeat yourself or restate the question.
7. Reply in the same language as the student's question."""

QUIZ_SYSTEM_PROMPT = (
    "You write accurate multiple-choice exam questions strictly from lecture material "
    "and reply with valid JSON only."
)


def require_api_key(x_api_key: str = Header(default="")):
    if not x_api_key:
        raise HTTPException(status_code=401, detail="Missing X-API-Key header.")

    if not secrets.compare_digest(x_api_key.encode(), LEARNMATE_API_KEY.encode()):
        raise HTTPException(status_code=401, detail="Invalid API key.")


def run_model(prompt, max_tokens, **options):
    if globals().get("model") is None or globals().get("tokenizer") is None:
        raise HTTPException(
            status_code=503,
            detail="The model is not loaded. Run the model-loading cell in the Kaggle notebook.",
        )

    try:
        return generate_text(prompt, max_tokens, **options)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise HTTPException(
            status_code=503,
            detail="The GPU ran out of memory. Try a shorter question or fewer quiz questions.",
        )


@app.exception_handler(Exception)
async def unexpected_error_handler(request: Request, error: Exception):
    traceback.print_exc()
    return JSONResponse(
        status_code=500,
        content={"detail": f"Internal server error ({type(error).__name__}). Check the Kaggle notebook logs."},
    )


class GenerateRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=12000)
    max_new_tokens: int = Field(default=512, ge=1, le=1024)


class ChatRequest(BaseModel):
    document_id: str = Field(min_length=1, max_length=64)
    question: str = Field(min_length=1, max_length=4000)

    @field_validator("document_id", "question")
    @classmethod
    def not_blank(cls, value):
        value = value.strip()

        if not value:
            raise ValueError("must not be blank")

        return value


class QuizRequest(BaseModel):
    document_id: str = Field(min_length=1, max_length=64)
    num_questions: int = Field(default=5, ge=1, le=10)
    topic: Optional[str] = Field(default=None, max_length=200)


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model": "Qwen3-4B-Instruct-2507",
        "model_loaded": globals().get("model") is not None,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "lectures_loaded": len(lecture_stores),
    }


@app.post("/generate", dependencies=[Depends(require_api_key)])
def generate(request: GenerateRequest):
    answer = run_model(request.prompt, request.max_new_tokens, do_sample=False)
    return {"answer": answer}


@app.post("/upload", dependencies=[Depends(require_api_key)])
def upload_lecture(file: UploadFile = File(...)):
    filename = file.filename or ""

    if not filename.lower().endswith(".pdf"):
        raise HTTPException(status_code=400, detail="Only PDF files are supported.")

    pdf_bytes = file.file.read(MAX_PDF_BYTES + 1)

    try:
        result = index_pdf_bytes(pdf_bytes, filename)
    except PdfProcessingError as error:
        raise HTTPException(status_code=error.status_code, detail=str(error))

    return {
        "status": "success",
        "message": "Lecture uploaded and indexed successfully.",
        **result,
    }


@app.post("/chat", dependencies=[Depends(require_api_key)])
def chat_with_lecture(request: ChatRequest):
    try:
        chunks, confidence = retrieve_chunks(request.document_id, request.question)
    except LookupError:
        raise HTTPException(status_code=404, detail=LECTURE_NOT_FOUND)

    response = {
        "document_id": request.document_id,
        "question": request.question,
        "confidence": confidence,
    }

    if not chunks:
        return {
            **response,
            "answer": (
                "I could not find this topic in your uploaded lecture. "
                "Try rephrasing the question using terms from the lecture."
            ),
            "sources": [],
            "cited_pages": [],
        }

    filename = lecture_info.get(request.document_id, {}).get("filename", "the lecture")
    weak_evidence_note = (
        "\nNote: these excerpts may be only loosely related to the question. "
        "If they do not answer it, say that the lecture does not cover it.\n"
        if confidence == "low" else ""
    )

    prompt = f"""LECTURE EXCERPTS (from "{filename}"):
{format_context(chunks)}

STUDENT QUESTION:
{request.question}
{weak_evidence_note}
Answer the question now, following the rules."""

    answer = run_model(
        prompt,
        400,
        system_prompt=TUTOR_SYSTEM_PROMPT,
        do_sample=False,
        repetition_penalty=1.05,
    )

    retrieved_pages = {item["page"] for item in chunks}
    cited_pages = sorted(
        {int(page) for page in re.findall(r"Page\s*(\d+)", answer)} & retrieved_pages
    )

    sources, seen_pages = [], set()

    for item in sorted(chunks, key=lambda c: c["chunk_index"]):
        if item["page"] in seen_pages:
            continue
        seen_pages.add(item["page"])
        sources.append({
            "page": item["page"],
            "score": item["score"],
            "excerpt": item["text"][:220],
        })

    return {**response, "answer": answer, "sources": sources, "cited_pages": cited_pages}


def parse_quiz(raw_output, allowed_pages):
    """Extract valid multiple-choice questions from the model's JSON output."""
    text = re.sub(r"```(?:json)?", "", raw_output)
    start, end = text.find("["), text.rfind("]")

    if start == -1 or end <= start:
        return []

    try:
        items = json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return []

    questions, seen = [], set()

    for item in items if isinstance(items, list) else []:
        if not isinstance(item, dict):
            continue

        question = str(item.get("question", "")).strip()
        options = item.get("options")

        if not question or question.lower() in seen:
            continue

        if not isinstance(options, list) or len(options) != 4:
            continue

        options = [re.sub(r"^\s*[A-Da-d][\).:]\s+", "", str(option)).strip() for option in options]

        if any(not option for option in options) or len({o.lower() for o in options}) != 4:
            continue

        answer_index = item.get("answer_index")

        if isinstance(answer_index, str):
            value = answer_index.strip().upper()
            answer_index = (
                int(value) if value.isdigit()
                else "ABCD".index(value) if len(value) == 1 and value in "ABCD"
                else None
            )

        if isinstance(answer_index, bool) or not isinstance(answer_index, int) or not 0 <= answer_index < 4:
            continue

        # Shuffle so the correct answer is not always in the same position.
        correct_option = options[answer_index]
        random.shuffle(options)

        try:
            page = int(item.get("page"))
        except (TypeError, ValueError):
            page = None

        seen.add(question.lower())
        questions.append({
            "question": question,
            "options": options,
            "correct_index": options.index(correct_option),
            "explanation": str(item.get("explanation", "")).strip(),
            "page": page if page in allowed_pages else None,
        })

    return questions


@app.post("/quiz", dependencies=[Depends(require_api_key)])
def generate_quiz(request: QuizRequest):
    topic = (request.topic or "").strip()

    try:
        chunks = retrieve_chunks(request.document_id, topic, top_k=6)[0] if topic else []
        chunks = chunks or sample_lecture_chunks(request.document_id)
    except LookupError:
        raise HTTPException(status_code=404, detail=LECTURE_NOT_FOUND)

    if not chunks:
        raise HTTPException(status_code=422, detail="The lecture has no text to build a quiz from.")

    count = request.num_questions
    prompt = f"""Write {count} multiple-choice questions that test understanding of the lecture excerpts below.

Requirements:
- Every question, correct answer and explanation must be directly supported by the excerpts. Do not use outside knowledge.
- Each question has exactly 4 options and exactly one correct option. Wrong options must be plausible but clearly wrong according to the excerpts.
- Cover different ideas; do not ask the same thing twice.
- Keep questions short and clear. Do not mention "the excerpt" or "the text".
- "answer_index" is the 0-based position of the correct option.
- "page" is the page number where the answer appears.

Return ONLY a JSON array with no other text, in exactly this format:
[{{"question": "...", "options": ["...", "...", "...", "..."], "answer_index": 0, "explanation": "One sentence on why the answer is correct.", "page": 3}}]

LECTURE EXCERPTS:
{format_context(chunks)}"""

    allowed_pages = {item["page"] for item in chunks}
    questions = []

    for attempt in range(2):
        raw_output = run_model(
            prompt,
            min(220 * count + 150, 2400),
            system_prompt=QUIZ_SYSTEM_PROMPT,
            do_sample=attempt > 0,  # retry with sampling if the first output was invalid
        )
        questions = parse_quiz(raw_output, allowed_pages)

        if questions:
            break

    if not questions:
        raise HTTPException(
            status_code=502,
            detail="The model did not return a valid quiz. Please try again.",
        )

    return {
        "document_id": request.document_id,
        "topic": topic or None,
        "questions": questions[:count],
        "source_pages": sorted(page for page in allowed_pages if page is not None),
    }


print("LearnMate API routes:", sorted({route.path for route in app.routes}))

LearnMate API routes: ['/chat', '/docs', '/docs/oauth2-redirect', '/generate', '/health', '/openapi.json', '/quiz', '/redoc', '/upload']


In [12]:
# Cell 8 — Start (or restart) the API server and the ngrok tunnel.
import socket
import threading
import time
import urllib.request

import uvicorn
from pyngrok import ngrok

PORT = 8000


def port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        return sock.connect_ex(("127.0.0.1", port)) == 0


# 1. Stop any server started earlier in this kernel (it would still serve the old app).
for server_name, thread_name in [("api_server", "api_server_thread"), ("server", "server_thread")]:
    old_server = globals().get(server_name)

    if isinstance(old_server, uvicorn.Server):
        old_server.should_exit = True
        old_thread = globals().get(thread_name)

        if old_thread is not None:
            old_thread.join(timeout=15)

        globals()[server_name] = None
        print("Stopped the previous API server.")

for _ in range(30):
    if not port_in_use(PORT):
        break
    time.sleep(0.5)
else:
    raise RuntimeError(
        f"Port {PORT} is still busy. Use Run > Restart & clear outputs, then run the cells again."
    )

# 2. Start the server in a background thread.
api_server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
api_server.install_signal_handlers = lambda: None

api_server_thread = threading.Thread(target=api_server.run, daemon=True, name="learnmate-api")
api_server_thread.start()

for _ in range(60):
    if api_server.started:
        break
    if not api_server_thread.is_alive():
        raise RuntimeError("The API server failed to start. See the error above.")
    time.sleep(0.5)
else:
    raise RuntimeError("The API server did not start within 30 seconds.")

# 3. Reuse the existing ngrok tunnel if there is one; otherwise open a new one.
ngrok.set_auth_token(NGROK_AUTHTOKEN)

tunnels = [t for t in ngrok.get_tunnels() if str(t.config.get("addr", "")).endswith(f":{PORT}")]

if tunnels:
    public_url = tunnels[0].public_url
    for extra_tunnel in tunnels[1:]:
        ngrok.disconnect(extra_tunnel.public_url)
    print("Reusing the existing ngrok tunnel.")
else:
    public_url = ngrok.connect(PORT, "http").public_url

with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=10) as response:
    print("Local health check:", response.read().decode())

print("\nLearnMate Qwen API URL:", public_url)
print("Health endpoint:", public_url + "/health")
print("API docs:", public_url + "/docs")
print("\nPut this URL in learnmate/.env.local as LEARNMATE_API_URL, then restart `npm run dev`.")

Local health check: {"status":"healthy","model":"Qwen3-4B-Instruct-2507","model_loaded":true,"gpu":"Tesla T4","lectures_loaded":0}

LearnMate Qwen API URL: https://keep-frostily-yoyo.ngrok-free.dev
Health endpoint: https://keep-frostily-yoyo.ngrok-free.dev/health
API docs: https://keep-frostily-yoyo.ngrok-free.dev/docs

Put this URL in learnmate/.env.local as LEARNMATE_API_URL, then restart `npm run dev`.


In [ ]:
# Cell 9_ Stop the API server and close the ngrok tunnel.
from pyngrok import ngrok

if globals().get("api_server") is not None:
    api_server.should_exit = True
    api_server_thread.join(timeout=15)
    api_server = None

ngrok.kill()
print("API server and ngrok tunnel stopped.")

__________________________________________________________________________________________________________________________________